In [0]:
def write_follower_followee_post_table(spark, run_date, db, table_names): 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{follower_followee_post} PARTITION (partition_date = '{run_date}')
              with user_base as (
                select distinct customer_id as user_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer
                where coalesce(to_date(from_unixtime(update_at / 1000)),to_date(from_unixtime(create_at / 1000))) <= '{run_date}'
                and to_date(from_unixtime(register_time / 1000)) <= '{run_date}'
                and ip_address is not null
                and delete_type = 0
                and `identity` != 2
                and agree_community_time is not null
                group by 1
              ),
              follower_followee as (
                select distinct customer_id as user_id, 
                passive_customer_id as followee_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer_follows
                where to_date(from_unixtime(follow_time/1000)) <= '{run_date}'
                and delete_at is null
              ),
              posts_published as (
                select distinct customer_id as user_id, 
                `id` as post_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where to_date(from_unixtime(post_time/1000)) = '{run_date}'
                and audit_status != 2 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and is_hide = 0  
                and publish_approval_status not in (0, 4) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
              )
              select ub.user_id, ff.followee_id, pp.post_id
              from user_base ub
              inner join follower_followee ff
              on ub.user_id = ff.user_id
              inner join posts_published pp
              on ff.followee_id = pp.user_id
              """.format(db = db,
                         follower_followee_post = table_names["follower_followee_post"],
                         run_date = run_date)
    )

In [0]:
import json
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
config_path = dbutils.widgets.get("config_path")

print("Input parameter run_date:{}".format(run_date))
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_follower_followee_post_table(spark, run_date, db, table_names)